# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My lane (or freestyle) and why

**Lane 4 — CTR / Engagement Opportunity Scoring.**

FlyRank pages don't just fail by ranking badly — a lot of them rank *fine* and still get skipped by
searchers. That's a click problem, not a visibility problem, and it's fixable without waiting months
for rankings to move (a title/meta rewrite can be tested in days). I'm choosing this lane provisionally
because the starter data already shows two things worth building on: (1) position tier clearly drives
expected CTR — mean CTR falls off sharply as pages move away from position 1 — so "low CTR" only means
something once you compare a page to *its own tier*, and (2) even inside the best tier (`page_1`),
there's real spread: a meaningful slice of pages sit at a fraction of their tier's typical CTR despite
getting real traffic. That's the opportunity set. I'm not locking this in — I can confirm or switch by
end of Week 4 — but it's the strongest early signal I found, and refresh/decline (Lane 2) looked noisier
in a first pass since "declining" doesn't say *why* a page is declining.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", 60)
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(f"rows: {len(df):,}  cols: {df.shape[1]}  clients: {df['client_id'].nunique()}")


rows: 30,000  cols: 44  clients: 32


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among pages that already rank reasonably well, which ones are under-capturing
clicks for their position — and are therefore worth an editor's time on the title, meta description,
or search snippet?

**Unit of analysis:** a page (`content_id`) — one row in the starter dataset already sits at this
grain (trailing 90-day metrics per pseudonymized content item). In the warehouse release the same
grain has to be built by rolling `fact_content_daily_performance` up over a chosen window per client.

**Output:** a ranked list of pages, each with an "opportunity score" (how far below its position
tier's expected CTR it sits, adjusted for volume), a reason code (e.g. "page_1 tier, CTR in bottom
half, 100+ impressions"), and a suggested action.

**Who acts, and what do they do:** a content/SEO editor works this list top-down instead of guessing
which of thousands of pages to open first. The action is concrete: rewrite the title tag or meta
description, improve the search snippet structure, or — if the gap turns out to be a real quality
issue — flag the page for a bigger content review instead of a metadata tweak.

**Cost of a wrong call:** two different mistakes, two different costs.
- *False positive* (flagging a page that isn't actually a CTR problem — just low-volume noise, or a
  tier that's misleading because of how the data rolls up): wastes editor hours on a page a rewrite
  won't help, and if it happens often enough, editors stop trusting the queue.
- *False negative* (missing a page genuinely under-capturing clicks): the page keeps losing clicks it
  could have earned, silently, for as long as it goes unreviewed.
  A minimum-volume filter is there specifically to make the first mistake less likely, at the cost of
  hiding some real-but-low-traffic opportunities.

**Why data/ML helps at all:** a single if-statement like "flag anything under 1% CTR" doesn't work,
because expected CTR depends heavily on position, and probably also on intent and content type — the
numbers below show roughly an 18x gap in average CTR between the best and worst position tiers. That's
too much structure to hand-write a fair threshold for every combination, but it's exactly the kind of
tangled-but-real pattern a simple, explainable scoring model (or even a well-built rule with per-tier
baselines) can capture honestly.


In [2]:
# no computation needed for this section — the numbers backing the claims above are in section 3


## 3. Quick look at the data (2-3 real numbers)

Loading the starter CSV (`data/raw/content_refresh_anonymized.csv`, 30,000 rows) and pulling three
numbers that motivate the lane: how much position tier alone moves CTR, how big the "page_1" opportunity
pool is, and how many of those pages look like real under-performers once a volume filter is applied.


In [3]:
# avg_position == 0 means "no data" per the data dictionary — exclude those rows before
# looking at CTR by position tier.
has_position = df[df["avg_position"] > 0].copy()
print(f"rows with real position data: {len(has_position):,} of {len(df):,}")

# Number 1 — CTR varies a lot by position tier, so "low CTR" only means something within a tier.
tier_ctr = (
    has_position.groupby("position_tier")["ctr"]
    .agg(mean_ctr="mean", median_ctr="median", n="count")
    .sort_values("mean_ctr", ascending=False)
)
print("\nMean/median CTR by position tier (ctr is a %, so 0.65 = 0.65%):")
print(tier_ctr.round(3))

best_tier = tier_ctr["mean_ctr"].idxmax()
worst_tier = tier_ctr["mean_ctr"].idxmin()
top_mean = tier_ctr.loc[best_tier, "mean_ctr"]
bottom_mean = tier_ctr.loc[worst_tier, "mean_ctr"]
print(f"\n{best_tier} mean CTR is ~{top_mean / bottom_mean:.1f}x {worst_tier}'s mean CTR "
      f"({top_mean:.2f}% vs {bottom_mean:.2f}%) — position alone is a huge driver of CTR.")

# Number 2 — page_1 is the single largest position tier: the biggest pool to search for opportunities.
page1 = has_position[has_position["position_tier"] == "page_1"]
print(f"\npage_1 tier size: {len(page1):,} pages "
      f"({len(page1) / len(has_position):.1%} of pages with position data)")

# Number 3 — inside page_1, how many pages sit well below their own tier's median CTR
# despite having enough impressions to trust the number (not just noise)?
median_ctr = page1["ctr"].median()
under_tier = page1[(page1["ctr"] < median_ctr * 0.5) & (page1["impressions_90d"] >= 100)]
print(f"page_1 median CTR: {median_ctr:.2f}%")
print(f"page_1 pages below half the tier median CTR AND with >=100 impressions_90d: "
      f"{len(under_tier):,} ({len(under_tier) / len(page1):.1%} of page_1)")


rows with real position data: 28,795 of 30,000

Mean/median CTR by position tier (ctr is a %, so 0.65 = 0.65%):
               mean_ctr  median_ctr      n
position_tier                             
top_3             2.764        0.00   1116
page_1            0.652        0.16  11814
striking          0.323        0.11   7304
page_3_5          0.222        0.03   7242
deep              0.150        0.00   1319

top_3 mean CTR is ~18.4x deep's mean CTR (2.76% vs 0.15%) — position alone is a huge driver of CTR.

page_1 tier size: 11,814 pages (41.0% of pages with position data)
page_1 median CTR: 0.16%
page_1 pages below half the tier median CTR AND with >=100 impressions_90d: 1,929 (16.3% of page_1)


## 4. Careful words: what I can and can't claim

**What this work can say:**
- *Observed:* which pages, at their observed position and volume, sit meaningfully below the CTR
  their own tier typically gets, over the trailing 90-day window in the data.
- *Directional / decision-support:* a ranked queue that points an editor to the most promising pages
  to review first, with a reason code, so their limited time goes to the highest-value guesses — not a
  guarantee that any specific page will improve.

**What this work will never claim:**
- *Not causal proof.* A page sitting below its tier's expected CTR doesn't prove the title or meta
  description is the cause — it could be intent mismatch, a misleading tier assignment, seasonality,
  or genuine noise at low volume. I'm not running a controlled experiment (no A/B test), so I can't
  say a rewrite *will* fix it — only that the page is worth a human look.
- *Not "predicting Google."* This never models or claims to model Google's ranking algorithm. It works
  entirely from observed clicks/impressions/position that already happened.
- *Not free of measurement quirks.* `avg_position = 0` means missing data, not "rank zero," and rate
  columns like `ctr` are already ×100 percentages — both are handled above, but any downstream numbers
  inherit whatever noise and pseudonymization already exist in this trailing-90-day snapshot.


In [4]:
# no additional computation needed for this section — it's a claims/limits statement, not a numeric one


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
